MACHINE LEARNING

This notebook contains Model Selection and Exploration.
1. Load  Processed Data.
2. Train/Test Split Data
3. Feature Scaling
4. train multiple baseline models with 5 fold cross validation - Here tree based models are chosen for the classification problem.
5. Compare their  evaluation metrics.
6. Hyperparameter Tuning on best model

In [119]:
# Locating the data
from pathlib import Path
# Directory where the notebook is running
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PROCESSED = PROJECT_ROOT/"data"/"processed"
# Path to the processed csv
data_processed_path = DATA_PROCESSED/"Telco-Customer-Churn-Processed.csv"

In [120]:
# Importing necessary libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, cross_val_score
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import classification_report, confusion_matrix

LOAD DATA

In [121]:
# Loading processed data
df = pd.read_csv(data_processed_path)
df.shape

(7021, 24)

TRAIN/TEST SPLIT

In [122]:
# Seperating features from target variable
X = df.drop(columns=['Churn'])
Y = df['Churn']
# Train/Test Split - 80/20
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)
print(f"Training data has {x_train.shape[0]} rows and {x_train.shape[1]} columns")
print(f"Testing data has {x_test.shape[0]} rows and {x_test.shape[1]} columns")

Training data has 5616 rows and 23 columns
Testing data has 1405 rows and 23 columns


FEATURE SCALING

To adjust the values of different data columns (features) so they all share a common numerical range or distribution. Here we are adjusting the continuous values columns.

In [123]:
# Find continuous numerical columns
continuous_columns = [
    col for col in x_train.columns if x_train[col].nunique() > 2
]
continuous_columns

['tenure', 'MonthlyCharges', 'TotalCharges']

StandardScaler adjust the values so that so each feature has a mean of 0 and a standard deviation of 1.

In [124]:
# StandardScaler - so that each feature mean is 0 and std is 1.
scaler = StandardScaler()

# Create copies of x_train and x_test data so that the original data is protected.
x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()

# fit training data and transform both.
x_train_scaled[continuous_columns] = scaler.fit_transform(
    x_train[continuous_columns]
)
x_test_scaled[continuous_columns] = scaler.transform(
    x_test[continuous_columns]
)
print("Automated Feature Scaling is done and data leakage is successfully blocked")


Automated Feature Scaling is done and data leakage is successfully blocked


THE 5 FOLD CROSS VALIDATION ON TREE BASED MODELS

During EDA, it showed high VIF scores, but we have not done any regularisation techniques. So training tree based models is the best choice here.

RandomForestClassifier, LightGBMClassifier, XGBoostClassifier

Evaluation Metrics: Accuracy, Recall, Precision, F1-Score and Time

Recall is the most important one as it predict the actual churners.

In [125]:
# dictionary with baseline models
models = {
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1)
}
#  list with evaluation metrics
scoring_metrics = ["accuracy", "precision", "recall", "f1"]
#  got a warning regading the " " in the feature names
x_train.columns= x_train.columns.str.replace(" ", "_")

# empty list to store the results
cv_results = []

for name, model in models.items():
    start_time = time.time()
    # cross validation
    cross_val_result = cross_validate(
        model, x_train, y_train, cv=5, scoring=scoring_metrics, return_train_score=True)
    results = {
        "Model": name,
        "Time": round((time.time() - start_time), 4),
        # Accuracy
        "Train_Accuracy": round((cross_val_result["train_accuracy"].mean()).item(), 4),
        "Test_Accuracy": round((cross_val_result["test_accuracy"].mean()).item(), 4),
        # Precision
        "Train_Precision": round((cross_val_result["train_precision"].mean()).item(), 4),
        "Test_Precision": round((cross_val_result["test_precision"].mean()).item(), 2),
        # Recall
        "Train_Recall": round((cross_val_result["train_recall"].mean()).item(), 4),
        "Test_Recall": round((cross_val_result["test_recall"].mean()).item(), 4),
        # f1Score
        "Train_F1": round((cross_val_result["train_f1"].mean()).item(), 4),
        "Test_F1": round((cross_val_result["test_f1"].mean()).item(), 4)
    }
    cv_results.append(results)


df_cv_results = pd.DataFrame(cv_results)
df_cv_results

cv_results


[{'Model': 'RandomForest',
  'Time': 0.5639,
  'Train_Accuracy': 0.9982,
  'Test_Accuracy': 0.786,
  'Train_Precision': 0.9973,
  'Test_Precision': 0.62,
  'Train_Recall': 0.996,
  'Test_Recall': 0.4835,
  'Train_F1': 0.9966,
  'Test_F1': 0.5434},
 {'Model': 'LightGBM',
  'Time': 0.2953,
  'Train_Accuracy': 0.8989,
  'Test_Accuracy': 0.7956,
  'Train_Precision': 0.8416,
  'Test_Precision': 0.64,
  'Train_Recall': 0.7611,
  'Test_Recall': 0.5327,
  'Train_F1': 0.7993,
  'Test_F1': 0.5795},
 {'Model': 'XGBoost',
  'Time': 0.2477,
  'Train_Accuracy': 0.948,
  'Test_Accuracy': 0.7812,
  'Train_Precision': 0.9164,
  'Test_Precision': 0.6,
  'Train_Recall': 0.884,
  'Test_Recall': 0.5111,
  'Train_F1': 0.8999,
  'Test_F1': 0.5525}]

LightGBM is  the overall best  according to the evaluation Metrics like Recall , F1 Score and Precision and Accuracy of test data. It also handles  overfitting better when compared to Random Forest and XGBoost. So we choose LightGBM. Currently, the precision is higher than Recall. This is a classification problem where we cannot miss a potential Churner cx, so we focus on increasing the recall value.

Why Baseline Recall is low?

Baseline models try to maximise global accuracy on an imbalanced data. Because of this imbalance, the models default to a conservative factory threshold of 0.5, which naturally results in a higher Precision but a lower Recall.

How will we improve Recall?

We have to improve the recall, so we will use scale_pos_weight parameter to penalise the model heavily if it miss a churner. Also we will optimise Decision Threshold(by lowering our decision threshold from 0.50 to 0.30 during the final prediction phase).

HYPERPARAMETER TUNING

LightGBM is optimised by hyperparameter tuning with the main focus is to improve recall. We use Optuna for hyperparameter tuning.

Optuna is a Bayesian Optimisation Framework. It uses TPE(Tree-Structured Parzen Estimators) and treats tuning like an intelligent AI search loop. It runs a random trial configuration, reviews the results and has a model to decide whether the result is improving or worsening. It learns and try to guess a better configuration allowing it to find best configuration in few trials(n_trials=30, in this case). 

In [126]:

def objective(trial):
    # setting the rule num_leaves < 2^max_depth
    max_depth = trial.suggest_int("max_depth", 3, 8)
    max_leaves_allowed = min(64, (2**max_depth) - 1)
    num_leaves = trial.suggest_int("num_leaves", 7, max_leaves_allowed)
    # parameters
    params= {
        # binary classification problem
        "objective" : "binary",
        # evaluation metrics optimised to trace binary probability errors
        "metric" : "binary_logloss",
        # gradient boosting decision tree
        "boosting_type" : "gbdt",
        # ensure reproducibility
        "random_state" : 42,
        # utilises all available cores in parallel
        "n_jobs" : -1,
        # silences logs
        "verbose" : -1,
        # restricts tree depth
        "max_depth" : max_depth,
        # restricts the maximum layout
        "num_leaves" : num_leaves,
        #  penalty for missing out a churner using Bayesian engine
        "scale_pos_weight" : trial.suggest_float("scale_pos_weight", 1.5, 4.0),
        # step size rate
        "learning_rate" : trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        # total trees built
        "n_estimators" : trial.suggest_int("n_estimators", 300, 800, step=50),
        # row sample percentage
        "subsample" : trial.suggest_float("subsample", 0.5, 1.0),
        # enables row sampling
        "bagging_freq" : 1,
        # column sample percentage
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        # L1 regularisation
        "reg_alpha" : trial.suggest_float("reg_alpha", 0.001, 2.0, log=True),
        # L2 regularisation
        "reg_lambda" : trial.suggest_float("reg_lambda", 0.001, 2.0, log=True)
    }
    # final model - LightGBM
    model = LGBMClassifier(**params)
    # cross validation
    cv_score_LGBM = cross_val_score(
        model, x_train, y_train, cv = 5, scoring="recall", n_jobs= 1
    )
    # returns the mean of all 5 recall score of each folds
    return cv_score_LGBM.mean()

# Start time
start_time = time.time()
# Ensure reproducibility
sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=30)
# end time
end_time = round((time.time() - start_time), 4)

print(f"Study Complete! Total optimisation time: {end_time}")
print(f"Peak Cross-Validated Recall Score: {study.best_value:.4f}")
print(f"Best Parameters: {study.best_params}")


[I 2026-09-24 03:07:40,885] A new study created in memory with name: no-name-54177dd3-18a7-4b51-a4a9-8389c6371622
[I 2026-09-24 03:07:41,527] Trial 0 finished with value: 0.7488215488215488 and parameters: {'max_depth': 5, 'num_leaves': 30, 'scale_pos_weight': 3.329984854528513, 'learning_rate': 0.060099747183803134, 'n_estimators': 350, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.7232250142857395, 'reg_lambda': 0.09644921218755018}. Best is trial 0 with value: 0.7488215488215488.
[I 2026-09-24 03:07:42,007] Trial 1 finished with value: 0.769023569023569 and parameters: {'max_depth': 7, 'num_leaves': 8, 'scale_pos_weight': 3.9247746304049858, 'learning_rate': 0.12106896936002161, 'n_estimators': 400, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 0.010099799910483412, 'reg_lambda': 0.05398047741548972}. Best is trial 1 with value: 0.769023569023569.
[I 2026-09-24 03:07:42,628] Trial 2 finished with value

Study Complete! Total optimisation time: 16.7922
Peak Cross-Validated Recall Score: 0.8606
Best Parameters: {'max_depth': 4, 'num_leaves': 10, 'scale_pos_weight': 3.96992056841012, 'learning_rate': 0.010679592221992763, 'n_estimators': 300, 'subsample': 0.8694816242977589, 'colsample_bytree': 0.5210773277288029, 'reg_alpha': 0.004046992198811344, 'reg_lambda': 0.027130564722099078}


The recall with the best parameters significantly improved on the training data. Best parameters are found. Use the same to fit the model using training data and predict churn using testing data.

In [127]:
# best parameters from hyperparameter tuning 
best_params = study.best_params
# final model
lgbm = LGBMClassifier(
    objective = "binary",
    metric = "binary_logloss",
    boosting_type = "gbdt",
    random_state = 42,
    n_jobs = -1,
    verbose = -1,
    **best_params
)

# Training
start_train = time.time()
lgbm.fit(x_train, y_train)
train_time = round((time.time()  - start_train), 4)
print(f" Training complete. Training time: {train_time}.")

# Testing prediction
start_pred = time.time()
test_probability = lgbm.predict_proba(x_test)[:, 1]

# compare the test probability with the preset threshold and convert it into 0 and 1
threshold = 0.5
y_pred = (test_probability >= threshold). astype(int)
pred_time = round((time.time()  - start_pred), 4)
print(f"Testing complete. Testing time: {pred_time}.")


 Training complete. Training time: 0.0788.
Testing complete. Testing time: 0.003.


In [133]:
# Evaluation metrics report
evaluation_metrics = classification_report(y_test, y_pred, digits=4)
print(f"Evaluation Metrics calculated \n{evaluation_metrics}")

Evaluation Metrics calculated 
              precision    recall  f1-score   support

           0     0.9298    0.6544    0.7682      1033
           1     0.4735    0.8629    0.6114       372

    accuracy                         0.7096      1405
   macro avg     0.7017    0.7587    0.6898      1405
weighted avg     0.8090    0.7096    0.7267      1405



In [134]:
# Confusion Metrics
confusion_matrix_plot = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix generated \n{confusion_matrix_plot}")

Confusion Matrix generated 
[[676 357]
 [ 51 321]]
